<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 30 · Autoencoder và VAE</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Nén MNIST bằng autoencoder và so với PCA, khử nhiễu, phát hiện ảnh lạ, rồi tự viết VAE (tái tham số hóa, KL divergence) và chấm ảnh sinh ra bằng một CNN.</p>
</div>

### Sau notebook này bạn sẽ

- Tự viết autoencoder và so sai số tái tạo với PCA.
- Dùng autoencoder để khử nhiễu và phát hiện ảnh lạ.
- Hiểu mẹo tái tham số hóa và kiểm công thức KL bằng mẫu.
- Sinh ảnh mới bằng VAE và đo chất lượng một cách công bằng.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> dưới 1 phút</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch30-autoencoder-vae/01-autoencoder/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · Autoencoder](#trang-1)
2. [Trang 2 · Không gian ẩn, khử nhiễu và phát hiện ảnh lạ](#trang-2)
3. [Trang 3 · VAE](#trang-3)
4. [Trang 4 · Sinh ảnh mới](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · Autoencoder

PCA và autoencoder trên MNIST (khoảng 40 giây trên CPU nhiều nhân).

*Đọc kèm trên web: [Trang 1 · Autoencoder](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch30-autoencoder-vae/01-autoencoder/)*

In [ ]:
import torch                                                # PyTorch
import torch.nn as nn                                       # các lớp mạng
from torchvision import datasets                            # MNIST
from torch.utils.data import TensorDataset, DataLoader      # chia lô
tr = datasets.MNIST("data", train=True, download=True)      # 60 000 ảnh train
te = datasets.MNIST("data", train=False, download=True)     # 10 000 ảnh test
Xtr = tr.data.float().div(255).view(-1, 784)                # mỗi ảnh thành vector 784 số
Xte = te.data.float().div(255).view(-1, 784)                # test
ytr, yte = tr.targets, te.targets                           # nhãn, chỉ dùng để kiểm tra
print(Xtr.shape, round(((Xte - Xtr.mean(0)) ** 2).mean().item(), 4))   # sai số khi đoán ảnh trung bình

Output mong đợi:

```text
torch.Size([60000, 784]) 0.0675
```

In [ ]:
from sklearn.decomposition import PCA                       # PCA của Chương 20
for k in (2, 32):                                           # hai mức nén
    pca = PCA(k, random_state=0).fit(Xtr.numpy())           # học trên train
    tai_tao = torch.tensor(pca.inverse_transform(pca.transform(Xte.numpy())))   # nén rồi bung
    print(k, round(((tai_tao - Xte) ** 2).mean().item(), 4))   # sai số tái tạo

Output mong đợi:

```text
2 0.0557
32 0.0168
```

In [ ]:
def tao_ae(k):                                              # mã hóa 784 -> k, giải mã k -> 784
    ma_hoa = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, k))   # bộ mã hóa
    giai_ma = nn.Sequential(nn.Linear(k, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid())   # bộ giải mã
    return ma_hoa, giai_ma                                  # hai mạng
def huan_luyen_ae(ma_hoa, giai_ma, so_epoch=5, nhieu=0.0):  # trả về sai số tái tạo trên test
    torch.manual_seed(1)                                    # thứ tự lô lặp lại được
    dl = DataLoader(TensorDataset(Xtr), batch_size=128, shuffle=True)   # lô 128 ảnh
    opt = torch.optim.Adam([*ma_hoa.parameters(), *giai_ma.parameters()], lr=1e-3)   # Adam cho cả hai mạng
    for ep in range(so_epoch):                              # mỗi epoch
        for (xb,) in dl:                                    # mỗi lô
            vao = (xb + nhieu * torch.randn_like(xb)).clamp(0, 1)   # nhieu=0: đầu vào sạch
            loss = ((giai_ma(ma_hoa(vao)) - xb) ** 2).mean()        # đích luôn là ảnh sạch
            opt.zero_grad()                                 # xóa gradient
            loss.backward()                                 # lượt ngược
            opt.step()                                      # cập nhật
    with torch.no_grad():                                   # đánh giá
        return round(((giai_ma(ma_hoa(Xte)) - Xte) ** 2).mean().item(), 4)   # sai số trên test
ae = {}                                                     # lưu hai autoencoder
for k in (2, 32):                                           # hai mức nén
    torch.manual_seed(0)                                    # cùng khởi tạo
    ae[k] = tao_ae(k)                                       # dựng
    print(k, sum(p.numel() for m in ae[k] for p in m.parameters()), huan_luyen_ae(*ae[k]))   # tham số, sai số

Output mong đợi:

```text
2 403730 0.0451
32 419120 0.0081
```

<a name="trang-2" id="trang-2"></a>

## Trang 2 · Không gian ẩn, khử nhiễu và phát hiện ảnh lạ

Không gian ẩn, khử nhiễu, phát hiện ảnh lạ.

*Đọc kèm trên web: [Trang 2 · Không gian ẩn, khử nhiễu và phát hiện ảnh lạ](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch30-autoencoder-vae/02-khong-gian-an/)*

In [ ]:
from sklearn.neighbors import KNeighborsClassifier          # kNN của Chương 14
ma2, giai2 = ae[2]                                          # autoencoder 2 số
with torch.no_grad():                                       # không cần gradient
    z_tr, z_te = ma2(Xtr), ma2(Xte)                         # mã 2 số của mọi ảnh
print([round(v, 2) for v in z_te.mean(0).tolist()], [round(v, 2) for v in z_te.std(0).tolist()])   # trung bình, độ lệch chuẩn
print([round(v, 1) for v in z_te.min(0).values.tolist()], [round(v, 1) for v in z_te.max(0).values.tolist()])   # nhỏ nhất, lớn nhất
pca2 = PCA(2, random_state=0).fit(Xtr.numpy())              # PCA 2 thành phần
for ten, a, b in (("AE", z_tr.numpy(), z_te.numpy()), ("PCA", pca2.transform(Xtr.numpy()), pca2.transform(Xte.numpy()))):   # hai cách nén
    print(ten, round(KNeighborsClassifier(5).fit(a, ytr.numpy()).score(b, yte.numpy()), 4))   # kNN đoán nhãn từ 2 số

Output mong đợi:

```text
[3.39, -2.82] [4.84, 3.5]
[-8.8, -12.9] [21.4, 9.5]
AE 0.5532
PCA 0.4237
```

In [ ]:
torch.manual_seed(0)                                        # cùng nhiễu
Xn = (Xte + 0.5 * torch.randn_like(Xte)).clamp(0, 1)        # ảnh test thêm nhiễu
ma32, giai32 = ae[32]                                       # autoencoder thường
torch.manual_seed(0)                                        # cùng khởi tạo
ma_kn, giai_kn = tao_ae(32)                                 # autoencoder khử nhiễu
print(huan_luyen_ae(ma_kn, giai_kn, nhieu=0.5))             # vào ảnh nhiễu, đích ảnh sạch
with torch.no_grad():                                       # đánh giá
    print(round(((Xn - Xte) ** 2).mean().item(), 4),        # không làm gì
          round(((giai32(ma32(Xn)) - Xte) ** 2).mean().item(), 4),   # autoencoder thường
          round(((giai_kn(ma_kn(Xn)) - Xte) ** 2).mean().item(), 4))   # autoencoder khử nhiễu

Output mong đợi:

```text
0.0218
0.1155 0.0713 0.0179
```

In [ ]:
def sai_so(X):                                              # sai số tái tạo của từng ảnh
    with torch.no_grad():                                   # không cần gradient
        return ((giai32(ma32(X)) - X) ** 2).mean(1)         # một số mỗi ảnh
nguong = sai_so(Xtr).quantile(0.99)                         # 1% ảnh train vượt ngưỡng
xoay = torch.rot90(Xte.view(-1, 28, 28), 1, dims=(1, 2)).reshape(-1, 784)   # ảnh test xoay 90 độ
print(round(nguong.item(), 4), round((sai_so(Xte) > nguong).float().mean().item(), 4),   # ngưỡng, báo nhầm
      round((sai_so(xoay) > nguong).float().mean().item(), 4))   # bắt được ảnh xoay
phat_hien = sai_so(xoay) > nguong                           # ảnh xoay nào bị đánh dấu
print([round(phat_hien[yte == c].float().mean().item(), 2) for c in range(10)])   # theo từng chữ số

Output mong đợi:

```text
0.0234 0.0079 0.4007
[0.2, 0.01, 0.59, 0.46, 0.31, 0.33, 0.62, 0.34, 0.77, 0.43]
```

<a name="trang-3" id="trang-3"></a>

## Trang 3 · VAE

Tái tham số hóa, KL divergence, VAE 2 chiều (khoảng 25 giây).

*Đọc kèm trên web: [Trang 3 · VAE](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch30-autoencoder-vae/03-vae/)*

In [ ]:
mu, log_var = torch.tensor([1.0, -2.0]), torch.tensor([0.0, -2.0])   # một phân phối chuẩn 2 chiều
torch.manual_seed(0)                                        # mẫu lặp lại được
z = mu + torch.exp(0.5 * log_var) * torch.randn(100000, 2)  # mẹo tái tham số hóa
print([round(v, 3) for v in z.mean(0).tolist()], [round(v, 3) for v in z.std(0).tolist()], round(torch.exp(torch.tensor(-1.0)).item(), 3))   # so với mu và sigma

Output mong đợi:

```text
[0.998, -2.002] [0.998, 0.369] 0.368
```

In [ ]:
from torch.distributions import Normal                      # mật độ phân phối chuẩn
kl_cong_thuc = 0.5 * (log_var.exp() + mu ** 2 - 1 - log_var)   # công thức đóng
q, p = Normal(mu, torch.exp(0.5 * log_var)), Normal(0.0, 1.0)  # q và N(0, 1)
kl_mc = (q.log_prob(z) - p.log_prob(z)).mean(0)             # định nghĩa, ước lượng bằng mẫu
print([round(v, 4) for v in kl_cong_thuc.tolist()], [round(v, 4) for v in kl_mc.tolist()])   # so hai cách

Output mong đợi:

```text
[0.5, 2.5677] [0.4976, 2.5691]
```

In [ ]:
import torch.nn.functional as F                             # hàm loss
class VAE(nn.Module):                                       # bien_phan=False: autoencoder thường
    def __init__(self, k, bien_phan=True):                  # k chiều ẩn
        super().__init__()                                  # bắt buộc
        self.ma_hoa = nn.Sequential(nn.Linear(784, 256), nn.ReLU())   # phần chung
        self.mu = nn.Linear(256, k)                         # trung bình
        self.log_var = nn.Linear(256, k)                    # log phương sai
        self.giai_ma = nn.Sequential(nn.Linear(k, 256), nn.ReLU(), nn.Linear(256, 784))   # ra logit
        self.bien_phan = bien_phan                          # có lấy mẫu và KL không
    def forward(self, x):                                   # lượt xuôi
        h = self.ma_hoa(x)                                  # đặc trưng
        mu, log_var = self.mu(h), self.log_var(h)           # tham số phân phối
        z = mu + torch.exp(0.5 * log_var) * torch.randn_like(mu) if self.bien_phan else mu   # tái tham số hóa
        return self.giai_ma(z), mu, log_var                 # logit ảnh, mu, log_var
def loss_vae(m, x):                                         # (tái tạo, KL) trên mỗi ảnh
    logit, mu, log_var = m(x)                               # lượt xuôi
    tai_tao = F.binary_cross_entropy_with_logits(logit, x, reduction="sum") / len(x)   # cộng trên điểm ảnh
    kl = 0.5 * (log_var.exp() + mu ** 2 - 1 - log_var).sum() / len(x) if m.bien_phan else torch.tensor(0.0)   # KL
    return tai_tao, kl                                      # hai số hạng
def huan_luyen_vae(m, so_epoch):                            # in sai số tái tạo và KL trên test
    torch.manual_seed(1)                                    # thứ tự lô
    dl = DataLoader(TensorDataset(Xtr), batch_size=128, shuffle=True)   # lô 128
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)         # Adam
    for ep in range(so_epoch):                              # mỗi epoch
        m.train()                                           # huấn luyện
        for (xb,) in dl:                                    # mỗi lô
            tai_tao, kl = loss_vae(m, xb)                   # hai số hạng
            opt.zero_grad()                                 # xóa gradient
            (tai_tao + kl).backward()                       # loss = tái tạo + KL
            opt.step()                                      # cập nhật
    m.eval()                                                # đánh giá
    with torch.no_grad():                                   # không cần gradient
        tai_tao, kl = loss_vae(m, Xte)                      # trên test
    print(round(tai_tao.item(), 1), round(kl.item(), 1))    # in ra
torch.manual_seed(0)                                        # cùng khởi tạo
vae2 = VAE(2)                                               # VAE 2 chiều
huan_luyen_vae(vae2, 10)                                    # 10 epoch
with torch.no_grad():                                       # mã của ảnh: dùng mu
    mu_tr, mu_te = vae2.mu(vae2.ma_hoa(Xtr)), vae2.mu(vae2.ma_hoa(Xte))   # train và test
print([round(v, 2) for v in mu_te.mean(0).tolist()], [round(v, 2) for v in mu_te.std(0).tolist()])   # quanh gốc, thang cỡ 1
print(round(KNeighborsClassifier(5).fit(mu_tr.numpy(), ytr.numpy()).score(mu_te.numpy(), yte.numpy()), 4))   # kNN trên mã

Output mong đợi:

```text
151.3 5.6
[0.31, 0.08] [1.17, 1.39]
0.6625
```

<a name="trang-4" id="trang-4"></a>

## Trang 4 · Sinh ảnh mới

Bộ chấm CNN, autoencoder và VAE 16 chiều, ba cách lấy mẫu (khoảng 2 phút).

*Đọc kèm trên web: [Trang 4 · Sinh ảnh mới](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch30-autoencoder-vae/04-sinh-anh/)*

In [ ]:
torch.manual_seed(0)                                        # cùng khởi tạo
cham = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # CNN của Chương 27
                     nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # khối thứ hai
                     nn.Flatten(), nn.Linear(32 * 7 * 7, 10))   # lớp cuối
torch.manual_seed(1)                                        # thứ tự lô
opt = torch.optim.Adam(cham.parameters(), lr=1e-3)          # Adam
for ep in range(2):                                         # 2 epoch
    for xb, yb in DataLoader(TensorDataset(Xtr.view(-1, 1, 28, 28), ytr), batch_size=64, shuffle=True):   # mỗi lô
        opt.zero_grad()                                     # xóa gradient
        F.cross_entropy(cham(xb), yb).backward()            # lượt ngược
        opt.step()                                          # cập nhật
cham.eval()                                                 # chế độ đánh giá
def cham_diem(anh):                                         # tỷ lệ ảnh CNN tin trên 90%, số ảnh mỗi lớp
    with torch.no_grad():                                   # không cần gradient
        p = cham(anh.view(-1, 1, 28, 28)).softmax(1)        # xác suất 10 lớp
    return round((p.max(1).values > 0.9).float().mean().item(), 3), torch.bincount(p.argmax(1), minlength=10).tolist()   # hai số đo
with torch.no_grad():                                       # độ chính xác của bộ chấm
    print(round((cham(Xte.view(-1, 1, 28, 28)).argmax(1) == yte).float().mean().item(), 4))   # trên test
print(cham_diem(Xte[:1000]))                                # mức trần: ảnh thật

Output mong đợi:

```text
0.9817
(0.927, [85, 125, 115, 111, 109, 84, 87, 98, 97, 89])
```

In [ ]:
torch.manual_seed(0)                                        # cùng khởi tạo
ae16 = VAE(16, bien_phan=False)                             # autoencoder thường: z = mu
huan_luyen_vae(ae16, 20)                                    # 20 epoch
torch.manual_seed(0)                                        # cùng khởi tạo
vae16 = VAE(16)                                             # VAE 16 chiều
huan_luyen_vae(vae16, 20)                                   # 20 epoch

Output mong đợi:

```text
72.1 0.0
82.7 23.2
```

In [ ]:
with torch.no_grad():                                       # không cần gradient
    z_ae = ae16.mu(ae16.ma_hoa(Xtr))                        # mã của 60 000 ảnh train
    z = torch.randn(1000, 16, generator=torch.Generator().manual_seed(0))   # 1000 điểm N(0, I)
    L = torch.linalg.cholesky(torch.cov(z_ae.T))            # Sigma = L L^T
    cach = {"AE, z ~ N(0, I)": z,                           # lấy mẫu bừa
            "AE, z theo phân phối chuẩn khớp với mã": z @ L.T + z_ae.mean(0),   # đúng trung bình, hiệp phương sai
            "VAE, z ~ N(0, I)": z}                          # VAE
    for ten, zz in cach.items():                            # ba cách
        m = vae16 if ten.startswith("VAE") else ae16        # mô hình tương ứng
        print(ten, cham_diem(torch.sigmoid(m.giai_ma(zz))))   # chấm ảnh sinh ra

Output mong đợi:

```text
AE, z ~ N(0, I) (0.007, [16, 79, 273, 73, 17, 365, 6, 168, 1, 2])
AE, z theo phân phối chuẩn khớp với mã (0.408, [72, 23, 136, 169, 87, 122, 87, 110, 135, 59])
VAE, z ~ N(0, I) (0.365, [64, 50, 140, 152, 101, 91, 82, 135, 108, 77])
```

<div style="border-left:6px solid #7A6A58;background-color:#F4F1EC;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#5C4E3F;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">THỬ SỬA</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>1.</b> Tăng số epoch của VAE lên 40. Tỷ lệ CNN tin chắc có tăng không?</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>2.</b> Đổi k thành 4 cho cả hai mô hình. Khoảng cách giữa AE khớp phân phối và VAE thay đổi thế nào?</p>
</div>

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Đoán trước, rồi mới mở đáp án.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch30-autoencoder-vae/05-bai-tap/)*

### Bài 30.1 · Autoencoder tuyến tính

Bỏ hết hàm kích hoạt, so sai số với PCA 32 thành phần (0,0168).
<details><summary>Xem đáp án</summary>

0,0170, sát PCA 0,0168: không có phi tuyến thì chỉ tìm được mặt phẳng của PCA. Code ở ô ngay dưới.
</details>

In [ ]:
torch.manual_seed(0)                                        # cùng khởi tạo
ma_tt, giai_tt = nn.Linear(784, 32), nn.Linear(32, 784)     # chỉ hai phép biến đổi tuyến tính
print(huan_luyen_ae(ma_tt, giai_tt))                        # 5 epoch như trang 1

Output mong đợi:

```text
0.017
```

### Bài 30.2 · KL bằng tay

KL của N(0, 1), N(3, 1), N(0; 0,01) so với N(0, 1).
<details><summary>Xem đáp án</summary>

0; 4,5; 1,808. Code ở ô ngay dưới.
</details>

In [ ]:
import math                                                 # log
for m, s in ((0.0, 1.0), (3.0, 1.0), (0.0, 0.1)):           # ba phân phối
    print(m, s, round(0.5 * (s ** 2 + m ** 2 - 1 - math.log(s ** 2)), 3))   # công thức đóng

Output mong đợi:

```text
0.0 1.0 0.0
3.0 1.0 4.5
0.0 0.1 1.808
```

### Bài 30.3 · Sinh riêng một chữ số

Lấy mẫu quanh mã trung bình của các ảnh số 7.
<details><summary>Xem đáp án</summary>

997 trên 1000 ảnh ra số 7, nhưng kém đa dạng. Code ở ô ngay dưới.
</details>

In [ ]:
with torch.no_grad():                                       # không cần gradient
    mu7 = vae16.mu(vae16.ma_hoa(Xtr[ytr == 7]))             # mã của mọi ảnh số 7
    z7 = mu7.mean(0) + 0.5 * mu7.std(0) * torch.randn(1000, 16, generator=torch.Generator().manual_seed(0))   # quanh trung bình
    anh7 = torch.sigmoid(vae16.giai_ma(z7))                 # giải mã
print(cham_diem(anh7))                                      # chấm

Output mong đợi:

```text
(0.974, [0, 0, 0, 0, 0, 2, 0, 997, 0, 1])
```